# 第 2 天作业 —— 用本地 Ollama 做网页摘要（可选 Selenium）

## 练习目标（理念）

把「网页抓取 + LLM 摘要」从云端 OpenAI 换成 **本地 Ollama** 开源模型（`llama3.2`），并学会在 JS 渲染站点上切换 **Selenium** 抓取。

## 和本课 Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Ollama OpenAI 兼容口 | `base_url=.../v1`，`api_key='ollama'` |
| 本地模型名 | `llama3.2`（可用 `llama3.2:1b` 省内存） |
| 网页抓取 | `scraper.fetch_website_contents` / `_selenium` |
| system / user messages | 讽刺风摘要助手 + 页面正文 |

## 怎么跑

1. 确保本机 Ollama 在跑：`http://localhost:11434`
2. `ollama pull llama3.2`（机器小可改 `llama3.2:1b`；别拉太大的 llama3.3 / llama4）
3. 需要 Selenium 时先在终端装好依赖并重启内核，再跑最后一格


In [ ]:
# ========== 导入：HTTP 探测、抓取器、展示、OpenAI 兼容客户端 ==========

# 导入标准库 os（本格主要占位；环境相关逻辑在 scraper / Ollama）
import os
# 导入 requests：探测 Ollama 是否在监听
import requests
# 从同目录 scraper 导入两种抓取：普通 requests/BeautifulSoup 与 Selenium（JS 站点）
from scraper import fetch_website_contents, fetch_website_contents_selenium
# 笔记本里用 Markdown 漂亮显示摘要
from IPython.display import Markdown, display
# OpenAI SDK：对接 Ollama 的 /v1 兼容接口（不是云端 OpenAI）
from openai import OpenAI

# 本地 Ollama 根地址（后面拼 /v1 给 SDK，或直接 GET 做健康检查）
OLLAMA_BASE_URL = "http://localhost:11434"


In [ ]:
# ========== 健康检查：Ollama 若没启动，这里会请求失败 ==========

# GET 根地址，看服务是否在跑；.content 取出响应体字节（有输出即说明通了）
requests.get(f'{OLLAMA_BASE_URL}').content


### 用 Ollama 拉取 `llama3.2`

若电脑内存较小，可改成 `llama3.2:1b`。

**不要**使用 `llama3.3` 或 `llama4`——它们对多数学习机来说太大了。


In [ ]:
# ========== 拉取模型到本机（shell magic；需本机已安装 ollama CLI） ==========

# ! 开头：在 notebook 里执行 shell；模型名字符串保持原文
!ollama pull llama3.2


In [ ]:
# ========== 创建指向本机 Ollama 的 OpenAI 兼容客户端 ==========

# base_url 走 /v1；api_key 对 Ollama 通常可为任意非空占位（这里用 'ollama'）
ollama = OpenAI(base_url=f'{OLLAMA_BASE_URL}/v1', api_key='ollama')


**可选（Selenium 抓取）：** 在终端运行：

```bash
uv pip install selenium webdriver-manager
```

然后**重启 Jupyter 内核**，让新包装进当前环境。

建议在终端装包，而不是在笔记本里直接跑 `uv` / `pip`，避免环境混乱。


In [ ]:
# ========== WebsiteSummarizer：抓页 → 组 messages → 本地 llama3.2 摘要 ==========

class WebsiteSummarizer:
    # 类属性：复用上面创建的 ollama 客户端
    client = ollama
    # system prompt 保持英文：讽刺风、忽略导航、直接回 Markdown（勿包代码块）
    system_prompt =  """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""
    # user 前缀保持英文：要求短摘要；若有新闻/公告也一并概括
    user_prompt_prefix =  """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.  
"""

    # url：目标站；use_selenium=True 时用浏览器渲染后再抽文本
    def __init__(self, url, use_selenium=False):
        # 保存 URL（便于调试；摘要本身主要靠 contents）
        self.url = url
        # JS 重站点走 Selenium；静态/SSR 站走普通 fetch
        fetcher = fetch_website_contents_selenium if use_selenium else fetch_website_contents
        # 立刻抓取页面正文
        self.contents = fetcher(url)
        # 同步调用模型生成摘要
        self.summary = self.summarize()
        # 原逻辑：__init__ 末尾 return display 的结果（副作用：构造时就展示）
        return self.display_summary()

    # 拼 Chat Completions 所需的 system + user 两条消息
    def messages_for(self):
        return [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f'{self.user_prompt_prefix}\n\n{self.contents}'}
        ]

    # 非流式调用本地 llama3.2，取完整 content
    def summarize(self):
        response = self.client.chat.completions.create(
            model="llama3.2",
            messages=self.messages_for()
        )
        return response.choices[0].message.content
      
    # 在笔记本单元格输出区渲染 Markdown 摘要
    def display_summary(self):
        display(Markdown(self.summary))   


In [ ]:
# ========== 演示 A：普通抓取 + 本地模型摘要 ==========

# 构造即抓取并展示；URL 保持原文
WebsiteSummarizer("https://www.andela.com")


In [ ]:
# ========== 演示 B：JS 站点 → Selenium 抓取 + 同一套摘要流程 ==========

# use_selenium=True 才会走 fetch_website_contents_selenium（需已安装依赖）
WebsiteSummarizer("https://dheerajmaddi.netlify.app/", use_selenium=True)
